# 3. Modeling — Sequential Funnel Experiments

**3-Stage Funnel Strategy:**
1. **Feature Comparison**: MFCC vs Chromagram vs MFCC+Chroma → LSTM on `raw_audio`
2. **Model Comparison**: RNN vs GRU vs LSTM vs BiLSTM vs Transformer → best feature on `raw_audio`
3. **Instrument Comparison**: all stems → best model + best feature

> Change `EXPERIMENT_STAGE` and related variables in Cell 1, then **Run All**.

In [276]:
# ============================================================
# Cell 1 — EXPERIMENT CONFIGURATION (Edit This Cell Only)
# ============================================================

# Stage: 'FEATURE' | 'MODEL' | 'INSTRUMENT'
EXPERIMENT_STAGE = 'INSTRUMENT'

# --- Stage 1: Feature Comparison ---
# Fixed: LSTM on raw_audio, vary features
STAGE1_INSTRUMENT = 'raw_audio'
STAGE1_MODEL = 'lstm'
STAGE1_FEATURES = ['mfcc', 'chromagram', 'mfcc_chroma']

# --- Stage 2: Model Comparison ---
# Fixed: best feature from Stage 1 on raw_audio, vary models
STAGE2_INSTRUMENT = 'raw_audio'
STAGE2_FEATURE = 'chromagram'  # ← UPDATE after Stage 1
STAGE2_MODELS = ['rnn', 'gru', 'lstm', 'bilstm', 'transformer']

# --- Stage 3: Instrument Comparison ---
# Fixed: best model + best feature, vary instruments
STAGE3_MODEL = 'lstm'    # ← UPDATE after Stage 2
STAGE3_FEATURE = 'chromagram'  # ← UPDATE after Stage 1
STAGE3_INSTRUMENTS = [
    'bass', 'guitar', 'guitar_piano', 'guitar_piano_bass',
    'no_vocals', 'piano', 'raw_audio', 'vocals',
]

# --- Hyperparameters ---
HIDDEN_SIZE = 128
NUM_LAYERS = 2
DROPOUT = 0.3
LEARNING_RATE = 1e-3
BATCH_SIZE = 8
EPOCHS = 100
VAL_RATIO = 0.2
RANDOM_SEED = 42

# --- Paths ---
PROCESSED_FILE = './processed/master_preprocessed.pkl'
OUTPUT_DIR = './outputs'

In [277]:
# ============================================================
# Cell 2 — Imports & Device Setup
# ============================================================
import json
import pickle
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.nn.functional as F

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

OUT_PATH = Path(OUTPUT_DIR)
OUT_PATH.mkdir(exist_ok=True)

print(f'Device: {DEVICE}')
print(f'Stage : {EXPERIMENT_STAGE}')

Device: mps
Stage : INSTRUMENT


In [278]:
# ============================================================
# Cell 3 — Load Preprocessed Data
# ============================================================
with open(PROCESSED_FILE, 'rb') as f:
    artifact = pickle.load(f)

master_data = artifact['data']
CLASS_NAMES = artifact['label_encoder_classes']
N_CLASSES = len(CLASS_NAMES)

print(f'Available keys: {list(master_data.keys())}')
print(f'N classes: {N_CLASSES}')

Available keys: ['bass__chromagram', 'bass__mfcc', 'bass__mfcc_chroma', 'guitar__chromagram', 'guitar__mfcc', 'guitar__mfcc_chroma', 'guitar_piano__chromagram', 'guitar_piano__mfcc', 'guitar_piano__mfcc_chroma', 'guitar_piano_bass__chromagram', 'guitar_piano_bass__mfcc', 'guitar_piano_bass__mfcc_chroma', 'no_vocals__chromagram', 'no_vocals__mfcc', 'no_vocals__mfcc_chroma', 'piano__chromagram', 'piano__mfcc', 'piano__mfcc_chroma', 'raw_audio__chromagram', 'raw_audio__mfcc', 'raw_audio__mfcc_chroma', 'vocals__chromagram', 'vocals__mfcc', 'vocals__mfcc_chroma']
N classes: 17


In [279]:
# ============================================================
# Cell 4 — Resolve Experiment Plan
# ============================================================
def build_experiment_plan():
    """Return list of (instrument, feature, model_key) tuples."""
    if EXPERIMENT_STAGE == 'FEATURE':
        return [
            (STAGE1_INSTRUMENT, feat, STAGE1_MODEL)
            for feat in STAGE1_FEATURES
        ]
    elif EXPERIMENT_STAGE == 'MODEL':
        return [
            (STAGE2_INSTRUMENT, STAGE2_FEATURE, mdl)
            for mdl in STAGE2_MODELS
        ]
    elif EXPERIMENT_STAGE == 'INSTRUMENT':
        return [
            (inst, STAGE3_FEATURE, STAGE3_MODEL)
            for inst in STAGE3_INSTRUMENTS
        ]
    else:
        raise ValueError(f'Unknown stage: {EXPERIMENT_STAGE}')

plan = build_experiment_plan()
print(f'Experiments to run ({len(plan)}):')
for i, (inst, feat, mdl) in enumerate(plan, 1):
    print(f'  {i}. {mdl:12s} on {inst:20s} with {feat}')

Experiments to run (8):
  1. lstm         on bass                 with chromagram
  2. lstm         on guitar               with chromagram
  3. lstm         on guitar_piano         with chromagram
  4. lstm         on guitar_piano_bass    with chromagram
  5. lstm         on no_vocals            with chromagram
  6. lstm         on piano                with chromagram
  7. lstm         on raw_audio            with chromagram
  8. lstm         on vocals               with chromagram


In [280]:
# ============================================================
# Cell 5 — Dataset & Collate
# ============================================================
class ChordSeqDataset(Dataset):
    """Wraps list of {X, y} sequence dicts into a PyTorch Dataset."""
    def __init__(self, sequences: List[dict]):
        self.X = [torch.from_numpy(s['X']) for s in sequences]
        self.y = [torch.from_numpy(s['y']) for s in sequences]

    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]


def collate_pad(batch):
    """Pad variable-length sequences. Label padding = -100 (ignored by CE loss)."""
    xs, ys = zip(*batch)
    lengths = torch.tensor([x.size(0) for x in xs])
    x_pad = pad_sequence(xs, batch_first=True, padding_value=0.0)
    y_pad = pad_sequence(ys, batch_first=True, padding_value=-100)
    return x_pad, y_pad, lengths

In [281]:
# ============================================================
# Cell 6 — Model Architectures
# ============================================================
class ChordRNN(nn.Module):
    def __init__(self, n_feat, n_cls, hidden=HIDDEN_SIZE, n_layers=NUM_LAYERS, drop=DROPOUT):
        super().__init__()
        self.rnn = nn.RNN(n_feat, hidden, n_layers, batch_first=True,
                          dropout=drop if n_layers > 1 else 0)
        self.fc = nn.Linear(hidden, n_cls)
    def forward(self, x, lengths=None):
        out, _ = self.rnn(x)
        return self.fc(out)

class ChordGRU(nn.Module):
    def __init__(self, n_feat, n_cls, hidden=HIDDEN_SIZE, n_layers=NUM_LAYERS, drop=DROPOUT):
        super().__init__()
        self.gru = nn.GRU(n_feat, hidden, n_layers, batch_first=True,
                          dropout=drop if n_layers > 1 else 0)
        self.fc = nn.Linear(hidden, n_cls)
    def forward(self, x, lengths=None):
        out, _ = self.gru(x)
        return self.fc(out)

class ChordLSTM(nn.Module):
    def __init__(self, n_feat, n_cls, hidden=HIDDEN_SIZE, n_layers=NUM_LAYERS, drop=DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(n_feat, hidden, n_layers, batch_first=True,
                            dropout=drop if n_layers > 1 else 0)
        self.fc = nn.Linear(hidden, n_cls)
    def forward(self, x, lengths=None):
        out, _ = self.lstm(x)
        return self.fc(out)

class ChordBiLSTM(nn.Module):
    def __init__(self, n_feat, n_cls, hidden=HIDDEN_SIZE, n_layers=NUM_LAYERS, drop=DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(n_feat, hidden, n_layers, batch_first=True,
                            dropout=drop if n_layers > 1 else 0, bidirectional=True)
        self.fc = nn.Linear(hidden * 2, n_cls)
    def forward(self, x, lengths=None):
        out, _ = self.lstm(x)
        return self.fc(out)

class ChordTransformer(nn.Module):
    def __init__(self, n_feat, n_cls, d_model=HIDDEN_SIZE, nhead=4,
                 n_layers=NUM_LAYERS, drop=DROPOUT):
        super().__init__()
        self.proj = nn.Linear(n_feat, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*2,
            dropout=drop, batch_first=True, activation='gelu'
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
        self.fc = nn.Linear(d_model, n_cls)
    def forward(self, x, lengths=None):
        z = self.proj(x)
        z = self.encoder(z)
        return self.fc(z)


MODEL_REGISTRY = {
    'rnn': ChordRNN,
    'gru': ChordGRU,
    'lstm': ChordLSTM,
    'bilstm': ChordBiLSTM,
    'transformer': ChordTransformer,
}

def build_model(key: str, n_feat: int, n_cls: int) -> nn.Module:
    return MODEL_REGISTRY[key](n_feat, n_cls).to(DEVICE)

In [282]:
# ============================================================
# Cell 7 — Training & Evaluation Loop
# ============================================================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, ignore_index=-100):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, ignore_index=self.ignore_index, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        
        if self.alpha is not None:
            safe_targets = targets.masked_fill(targets == self.ignore_index, 0)
            alpha_t = self.alpha.gather(0, safe_targets)
            focal_loss = focal_loss * alpha_t
            
        valid_mask = targets != self.ignore_index
        if valid_mask.sum() > 0:
            return focal_loss[valid_mask].mean()
        else:
            return focal_loss.sum() * 0.0

def masked_accuracy(logits, targets):
    pred = logits.argmax(dim=-1)
    mask = targets != -100
    if mask.sum() == 0:
        return 0.0
    return (pred[mask] == targets[mask]).float().mean().item()


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, total_acc, n = 0., 0., 0
    for xb, yb, lens in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb, lens)
        loss = criterion(logits.view(-1, logits.size(-1)), yb.view(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        total_acc += masked_accuracy(logits, yb)
        n += 1
    return total_loss / max(n,1), total_acc / max(n,1)


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_acc, n = 0., 0., 0
    for xb, yb, lens in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb, lens)
        loss = criterion(logits.view(-1, logits.size(-1)), yb.view(-1))
        total_loss += loss.item()
        total_acc += masked_accuracy(logits, yb)
        n += 1
    return total_loss / max(n,1), total_acc / max(n,1)


def run_experiment(data_key, model_key):
    """Train one model on one dataset and return results dict."""
    entry = master_data[data_key]
    seqs = list(entry['sequences'])  # copy
    np.random.shuffle(seqs)

    n_val = max(1, int(len(seqs) * VAL_RATIO))
    val_seqs, train_seqs = seqs[:n_val], seqs[n_val:]

    train_loader = DataLoader(ChordSeqDataset(train_seqs), batch_size=BATCH_SIZE,
                              shuffle=True, collate_fn=collate_pad)
    val_loader = DataLoader(ChordSeqDataset(val_seqs), batch_size=BATCH_SIZE,
                            shuffle=False, collate_fn=collate_pad)

    n_feat = len(entry['feature_columns'])
    model = build_model(model_key, n_feat, N_CLASSES)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    criterion = FocalLoss(gamma=2.0, ignore_index=-100)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = -1.0

    for ep in range(1, EPOCHS + 1):
        tl, ta = train_one_epoch(model, train_loader, optimizer, criterion)
        vl, va = evaluate(model, val_loader, criterion)
        history['train_loss'].append(tl)
        history['train_acc'].append(ta)
        history['val_loss'].append(vl)
        history['val_acc'].append(va)
        if va > best_val_acc:
            best_val_acc = va
        if ep % 10 == 0:
            print(f'    Ep {ep:3d}/{EPOCHS} │ TrL={tl:.4f} TrA={ta:.4f} │ VaL={vl:.4f} VaA={va:.4f}')

    return {
        'best_val_acc': best_val_acc,
        'history': history,
        'n_train': len(train_seqs),
        'n_val': len(val_seqs),
        'n_features': n_feat,
    }

In [283]:
# ============================================================
# Cell 8 — Execute All Experiments
# ============================================================
all_results = []

for inst, feat, mdl in plan:
    data_key = f'{inst}__{feat}'
    if data_key not in master_data:
        print(f'⚠ Skipping {data_key} (not in preprocessed data)')
        continue

    print(f'\n{"="*60}')
    print(f'  {mdl.upper()} × {inst} × {feat}')
    print(f'{"="*60}')

    result = run_experiment(data_key, mdl)
    result.update({'instrument': inst, 'feature': feat, 'model': mdl})
    all_results.append(result)

    print(f'  → Best Val Accuracy: {result["best_val_acc"]:.4f}')

# Save results
out_file = OUT_PATH / f'results_stage_{EXPERIMENT_STAGE.lower()}.json'

# Strip history for JSON (keep it lean)
json_results = []
for r in all_results:
    jr = {k: v for k, v in r.items() if k != 'history'}
    jr['final_train_loss'] = r['history']['train_loss'][-1]
    jr['final_val_acc'] = r['history']['val_acc'][-1]
    json_results.append(jr)

with open(out_file, 'w') as f:
    json.dump(json_results, f, indent=2, default=str)

# Also save full history as pkl for analysis notebook
with open(OUT_PATH / f'results_stage_{EXPERIMENT_STAGE.lower()}_full.pkl', 'wb') as f:
    pickle.dump(all_results, f)

print(f'\n\nResults saved to {out_file}')
print(f'\n--- SUMMARY ---')
for r in json_results:
    print(f'  {r["model"]:12s} │ {r["instrument"]:20s} │ {r["feature"]:15s} │ best_acc={r["best_val_acc"]:.4f}')


  LSTM × bass × chromagram
    Ep  10/100 │ TrL=1.4194 TrA=0.2659 │ VaL=1.4435 VaA=0.2304
    Ep  20/100 │ TrL=1.3372 TrA=0.3220 │ VaL=1.4182 VaA=0.2454
    Ep  30/100 │ TrL=1.1219 TrA=0.4242 │ VaL=1.3983 VaA=0.2373
    Ep  40/100 │ TrL=0.8914 TrA=0.5391 │ VaL=1.3042 VaA=0.3001
    Ep  50/100 │ TrL=0.7231 TrA=0.6192 │ VaL=1.2615 VaA=0.3389
    Ep  60/100 │ TrL=0.5616 TrA=0.7089 │ VaL=1.2556 VaA=0.3568
    Ep  70/100 │ TrL=0.4379 TrA=0.7672 │ VaL=1.2854 VaA=0.3572
    Ep  80/100 │ TrL=0.3845 TrA=0.7825 │ VaL=1.2895 VaA=0.3681
    Ep  90/100 │ TrL=0.3173 TrA=0.8173 │ VaL=1.3012 VaA=0.3648
    Ep 100/100 │ TrL=0.2510 TrA=0.8534 │ VaL=1.3351 VaA=0.3797
  → Best Val Accuracy: 0.3978

  LSTM × guitar × chromagram
    Ep  10/100 │ TrL=1.3940 TrA=0.2692 │ VaL=1.5343 VaA=0.2365
    Ep  20/100 │ TrL=1.2835 TrA=0.3519 │ VaL=1.4949 VaA=0.2920
    Ep  30/100 │ TrL=1.0208 TrA=0.4525 │ VaL=1.3667 VaA=0.3350
    Ep  40/100 │ TrL=0.7574 TrA=0.5765 │ VaL=1.2443 VaA=0.4223
    Ep  50/100 │ TrL=0.5946 Tr